In [ ]:
#cell1

!pip -q install -U "transformers>=4.51.0" "sentence-transformers>=2.7.0" "chromadb>=1.0.0" "tqdm" "numpy" "pandas"

In [ ]:
#cell2

import os
import json
import time
import gc
import shutil
from pathlib import Path
from typing import Dict, List, Any, Optional

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import chromadb
from sentence_transformers import SentenceTransformer

In [ ]:
#cell3

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
#cell4

BASE_DIR = Path("/content/drive/MyDrive/final_project/RAG")

MODEL_NAME = "Qwen/Qwen3-Embedding-8B"
EXPECTED_EMBEDDING_DIM = 4096

# Batch size is intentionally conservative because the model is large.
# On A100 with enough memory, you can try 24 or 32.
ENCODE_BATCH_SIZE = 16

# This is only for the retrieval sanity-check cell.
DEFAULT_TOP_K = 6

# If True, old Chroma folders will be removed before rebuilding.
REBUILD_DATABASES = True

# Use None for full build. Use a small number like 100 for debugging.
MAX_CHUNKS_PER_DATASET = None

DATASETS = {
    "hotpotqa": {
        "json_path": BASE_DIR / "hotpotqa_docs_chunks.json",
        "db_dir": BASE_DIR / "hotpotqa",
        "collection_name": "chunks",
    },
    "2wikimultihopqa": {
        "json_path": BASE_DIR / "2wikimultihopqa_docs_chunks.json",
        "db_dir": BASE_DIR / "2wikimultihopqa",
        "collection_name": "chunks",
    },
}

print("Base directory:", BASE_DIR)
for dataset_name, cfg in DATASETS.items():
    print(dataset_name, "json:", cfg["json_path"])
    print(dataset_name, "db:", cfg["db_dir"])

Base directory: /content/drive/MyDrive/final_project/RAG
hotpotqa json: /content/drive/MyDrive/final_project/RAG/hotpotqa_docs_chunks.json
hotpotqa db: /content/drive/MyDrive/final_project/RAG/hotpotqa
2wikimultihopqa json: /content/drive/MyDrive/final_project/RAG/2wikimultihopqa_docs_chunks.json
2wikimultihopqa db: /content/drive/MyDrive/final_project/RAG/2wikimultihopqa


In [ ]:
#cell5

assert BASE_DIR.exists(), f"BASE_DIR does not exist: {BASE_DIR}"

for dataset_name, cfg in DATASETS.items():
    assert cfg["json_path"].exists(), f"Missing JSON file for {dataset_name}: {cfg['json_path']}"

if REBUILD_DATABASES:
    for dataset_name, cfg in DATASETS.items():
        if cfg["db_dir"].exists():
            shutil.rmtree(cfg["db_dir"])
            print(f"Removed old DB folder for {dataset_name}: {cfg['db_dir']}")

for dataset_name, cfg in DATASETS.items():
    cfg["db_dir"].mkdir(parents=True, exist_ok=True)
    print(f"Ready DB folder for {dataset_name}: {cfg['db_dir']}")

Ready DB folder for hotpotqa: /content/drive/MyDrive/final_project/RAG/hotpotqa
Ready DB folder for 2wikimultihopqa: /content/drive/MyDrive/final_project/RAG/2wikimultihopqa


In [ ]:
#cell6

def safe_int(value: Any, default: int = -1) -> int:
    """Convert a value to int safely."""
    try:
        return int(value)
    except (TypeError, ValueError):
        return default


def make_document_text(record: Dict[str, Any]) -> str:
    """Build the exact text that will be embedded and stored as the Chroma document."""
    title = str(record.get("Title", "")).strip()
    text = str(record.get("Text", "")).strip()
    return f"Title: {title}\nText: {text}"


def make_metadata(record: Dict[str, Any], dataset_name: str) -> Dict[str, Any]:
    """Build Chroma metadata for one chunk."""
    paragraph_ids = record.get("Paragraph_id", [])
    if paragraph_ids is None:
        paragraph_ids = []

    return {
        "dataset": dataset_name,
        "chunk_id": str(record["Chunk_id"]),
        "title": str(record.get("Title", "")),
        "paragraph_id_json": json.dumps(paragraph_ids, ensure_ascii=False),
        "token_count": safe_int(record.get("Token_count", -1), default=-1),
    }


def load_and_validate_chunks(
    json_path: Path,
    dataset_name: str,
    max_chunks: Optional[int] = None
) -> List[Dict[str, Any]]:
    """Load and validate chunk records from a JSON list."""
    with open(json_path, "r", encoding="utf-8") as f:
        chunks = json.load(f)

    if not isinstance(chunks, list):
        raise ValueError(f"{dataset_name}: JSON root must be a list.")

    if max_chunks is not None:
        chunks = chunks[:max_chunks]

    required_keys = {"Chunk_id", "Title", "Text"}
    ids = []

    for i, record in enumerate(chunks):
        if not isinstance(record, dict):
            raise ValueError(f"{dataset_name}: record {i} is not a dictionary.")

        missing = required_keys - set(record.keys())
        if missing:
            raise ValueError(f"{dataset_name}: record {i} is missing keys: {missing}")

        chunk_id = str(record["Chunk_id"]).strip()
        if not chunk_id:
            raise ValueError(f"{dataset_name}: record {i} has an empty Chunk_id.")

        ids.append(chunk_id)

    duplicate_ids = pd.Series(ids).value_counts()
    duplicate_ids = duplicate_ids[duplicate_ids > 1]

    if len(duplicate_ids) > 0:
        raise ValueError(
            f"{dataset_name}: duplicate Chunk_id values found. "
            f"Examples: {duplicate_ids.head(10).to_dict()}"
        )

    print(f"{dataset_name}: loaded {len(chunks):,} chunks.")
    return chunks

In [ ]:
#cell7

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

model = SentenceTransformer(MODEL_NAME, device=device)

probe_text = ["Title: Test\nText: This is a short test document."]
probe_embedding = model.encode(
    probe_text,
    convert_to_numpy=True,
    show_progress_bar=False
)

print("Probe embedding shape:", probe_embedding.shape)

assert probe_embedding.shape[1] == EXPECTED_EMBEDDING_DIM, (
    f"Expected embedding dimension {EXPECTED_EMBEDDING_DIM}, "
    f"but got {probe_embedding.shape[1]}"
)

del probe_embedding
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

Device: cuda
GPU: NVIDIA A100-SXM4-40GB


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.3k [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Probe embedding shape: (1, 4096)


In [ ]:
#cell8

def get_chroma_collection(db_dir: Path, collection_name: str):
    """Create or load a persistent Chroma collection with cosine distance."""
    client = chromadb.PersistentClient(path=str(db_dir))

    try:
        collection = client.get_or_create_collection(
            name=collection_name,
            configuration={"hnsw": {"space": "cosine"}}
        )
    except TypeError:
        # Compatibility fallback for older Chroma versions.
        collection = client.get_or_create_collection(
            name=collection_name,
            metadata={"hnsw:space": "cosine"}
        )

    return client, collection

In [ ]:
#cell9

def build_vector_database_for_dataset(
    dataset_name: str,
    cfg: Dict[str, Any],
    model: SentenceTransformer,
    encode_batch_size: int = 16,
    max_chunks: Optional[int] = None
) -> Dict[str, Any]:
    """Build one persistent Chroma vector database for one dataset."""
    chunks = load_and_validate_chunks(
        json_path=cfg["json_path"],
        dataset_name=dataset_name,
        max_chunks=max_chunks
    )

    client, collection = get_chroma_collection(
        db_dir=cfg["db_dir"],
        collection_name=cfg["collection_name"]
    )

    start_time = time.time()
    total_added = 0

    for start_idx in tqdm(
        range(0, len(chunks), encode_batch_size),
        desc=f"Embedding + indexing {dataset_name}"
    ):
        batch_records = chunks[start_idx:start_idx + encode_batch_size]

        batch_ids = [str(record["Chunk_id"]) for record in batch_records]
        batch_documents = [make_document_text(record) for record in batch_records]
        batch_metadatas = [make_metadata(record, dataset_name) for record in batch_records]

        batch_embeddings = model.encode(
            batch_documents,
            batch_size=len(batch_documents),
            convert_to_numpy=True,
            show_progress_bar=False
        )

        if batch_embeddings.shape[1] != EXPECTED_EMBEDDING_DIM:
            raise ValueError(
                f"{dataset_name}: expected dim {EXPECTED_EMBEDDING_DIM}, "
                f"got {batch_embeddings.shape[1]}"
            )

        # Chroma stores embeddings as numeric vectors. This does not change model loading dtype.
        batch_embeddings = np.asarray(batch_embeddings, dtype=np.float32)

        collection.add(
            ids=batch_ids,
            embeddings=batch_embeddings.tolist(),
            documents=batch_documents,
            metadatas=batch_metadatas
        )

        total_added += len(batch_records)

        del batch_embeddings
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    elapsed_seconds = time.time() - start_time
    final_count = collection.count()

    manifest = {
        "dataset": dataset_name,
        "source_json": str(cfg["json_path"]),
        "db_dir": str(cfg["db_dir"]),
        "collection_name": cfg["collection_name"],
        "model_name": MODEL_NAME,
        "embedding_dimension": EXPECTED_EMBEDDING_DIM,
        "similarity_space": "cosine",
        "document_format": "Title: {Title}\\nText: {Text}",
        "num_input_chunks": len(chunks),
        "collection_count": final_count,
        "encode_batch_size": encode_batch_size,
        "elapsed_seconds": elapsed_seconds,
        "rebuilt_database": REBUILD_DATABASES,
    }

    manifest_path = cfg["db_dir"] / "build_manifest.json"
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)

    print(f"{dataset_name}: collection count = {final_count:,}")
    print(f"{dataset_name}: manifest saved to {manifest_path}")

    return manifest

In [10]:
#cell10

build_manifests = []

for dataset_name, cfg in DATASETS.items():
    manifest = build_vector_database_for_dataset(
        dataset_name=dataset_name,
        cfg=cfg,
        model=model,
        encode_batch_size=ENCODE_BATCH_SIZE,
        max_chunks=MAX_CHUNKS_PER_DATASET
    )
    build_manifests.append(manifest)

build_summary_df = pd.DataFrame(build_manifests)
display(build_summary_df)

hotpotqa: loaded 35,029 chunks.


Embedding + indexing hotpotqa:   0%|          | 0/2190 [00:00<?, ?it/s]

hotpotqa: collection count = 35,029
hotpotqa: manifest saved to /content/drive/MyDrive/final_project/RAG/hotpotqa/build_manifest.json
2wikimultihopqa: loaded 12,685 chunks.


Embedding + indexing 2wikimultihopqa:   0%|          | 0/793 [00:00<?, ?it/s]

2wikimultihopqa: collection count = 12,685
2wikimultihopqa: manifest saved to /content/drive/MyDrive/final_project/RAG/2wikimultihopqa/build_manifest.json


,dataset,source_json,db_dir,collection_name,model_name,embedding_dimension,similarity_space,document_format,num_input_chunks,collection_count,encode_batch_size,elapsed_seconds,rebuilt_database
0,hotpotqa,/content/drive/MyDrive/final_project/RAG/hotpo...,/content/drive/MyDrive/final_project/RAG/hotpotqa,chunks,Qwen/Qwen3-Embedding-8B,4096,cosine,Title: {Title}\nText: {Text},35029,35029,16,3152.984844,True
1,2wikimultihopqa,/content/drive/MyDrive/final_project/RAG/2wiki...,/content/drive/MyDrive/final_project/RAG/2wiki...,chunks,Qwen/Qwen3-Embedding-8B,4096,cosine,Title: {Title}\nText: {Text},12685,12685,16,1079.407422,True


In [11]:
#cell11

def load_collection(dataset_name: str):
    """Load an existing Chroma collection for a dataset."""
    cfg = DATASETS[dataset_name]
    client = chromadb.PersistentClient(path=str(cfg["db_dir"]))
    collection = client.get_collection(name=cfg["collection_name"])
    return collection


for dataset_name in DATASETS.keys():
    collection = load_collection(dataset_name)
    sample = collection.get(
        limit=1,
        include=["documents", "metadatas"]
    )

    print("=" * 80)
    print("Dataset:", dataset_name)
    print("Count:", collection.count())
    print("Sample ID:", sample["ids"][0] if sample["ids"] else None)
    print("Sample metadata:", sample["metadatas"][0] if sample["metadatas"] else None)
    print("Sample document preview:")
    print(sample["documents"][0][:500] if sample["documents"] else None)

Dataset: hotpotqa
Count: 35029
Sample ID: hotpotqa_chunk_00000001
Sample metadata: {'dataset': 'hotpotqa', 'token_count': 411, 'paragraph_id_json': '[1, 2, 3, 4, 5]', 'title': 'Meet Corliss Archer', 'chunk_id': 'hotpotqa_chunk_00000001'}
Sample document preview:
Title: Meet Corliss Archer
Text: Meet Corliss Archer, a program from radio's Golden Age, ran from January 7, 1943 to September 30, 1956. Although it was CBS's answer to NBC's popular "A Date with Judy", it was also broadcast by NBC in 1948 as a summer replacement for "The Bob Hope Show". From October 3, 1952 to June 26, 1953, it aired on ABC, finally returning to CBS. Despite the program's long run, fewer than 24 episodes are known to exist.
Priscilla Lyon and Janet Waldo successively portrayed 
Dataset: 2wikimultihopqa
Count: 12685
Sample ID: 2wikimultihopqa_chunk_00000001
Sample metadata: {'paragraph_id_json': '[1, 2, 3, 4]', 'title': 'Calloway County High School', 'chunk_id': '2wikimultihopqa_chunk_00000001', 'dataset': '2wi

In [12]:
#cell12

def embed_query(question: str) -> np.ndarray:
    """Embed a user question using the Qwen query prompt."""
    query_embedding = model.encode(
        [question],
        prompt_name="query",
        convert_to_numpy=True,
        show_progress_bar=False
    )

    if query_embedding.shape[1] != EXPECTED_EMBEDDING_DIM:
        raise ValueError(
            f"Expected query dim {EXPECTED_EMBEDDING_DIM}, "
            f"got {query_embedding.shape[1]}"
        )

    return np.asarray(query_embedding, dtype=np.float32)


def retrieve_top_k_chunks(
    dataset_name: str,
    question: str,
    top_k: int = 6
) -> pd.DataFrame:
    """Retrieve top-k relevant chunks from one dataset Chroma DB."""
    collection = load_collection(dataset_name)
    query_embedding = embed_query(question)

    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )

    rows = []
    for rank, (chunk_id, document, metadata, distance) in enumerate(
        zip(
            results["ids"][0],
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0]
        ),
        start=1
    ):
        rows.append({
            "rank": rank,
            "chunk_id": chunk_id,
            "title": metadata.get("title"),
            "cosine_distance": float(distance),
            "cosine_similarity": 1.0 - float(distance),
            "document": document,
            "metadata": metadata,
        })

    return pd.DataFrame(rows)

In [13]:
#cell13

hotpotqa_test_question = "What is Meet Corliss Archer?"
hotpotqa_results = retrieve_top_k_chunks(
    dataset_name="hotpotqa",
    question=hotpotqa_test_question,
    top_k=DEFAULT_TOP_K
)

display(hotpotqa_results[["rank", "chunk_id", "title", "cosine_distance", "cosine_similarity"]])
print(hotpotqa_results.loc[0, "document"][:1000])

,rank,chunk_id,title,cosine_distance,cosine_similarity
0,1,hotpotqa_chunk_00000022,Meet Corliss Archer (TV series),0.237972,0.762028
1,2,hotpotqa_chunk_00000001,Meet Corliss Archer,0.243914,0.756086
2,3,hotpotqa_chunk_00000002,Meet Corliss Archer,0.251356,0.748644
3,4,hotpotqa_chunk_00000019,Janet Waldo,0.388814,0.611186
4,5,hotpotqa_chunk_00000025,A Kiss for Corliss,0.408320,0.591680
5,6,hotpotqa_chunk_00000026,Kiss and Tell (1945 film),0.443683,0.556317


Title: Meet Corliss Archer (TV series)
Text: Meet Corliss Archer is an American television sitcom that aired on CBS (July 13, 1951 - August 10, 1951) and in syndication via the Ziv Company from April to December 1954. The program was an adaptation of the radio series of the same name, which was based on a series of short stories by F. Hugh Herbert.
Corliss Archer is a lovable blonde teenager delicately balancing her high-school life and relationship with goofy boyfriend Dexter Franklin, and her homelife with parents Harry and Janet Archer.
Source: "Encyclopedia of Television Shows, 1925 through 2010"
The syndicated version of "Meet Corliss Archer" was executive produced by Frederick W. Ziv, and produced by ZIV Television Programs.
The series, which is public domain, is occasionally still repeated in the United States today, usually on small over-the-air networks and cable channels.
The series has also appeared on DVDs by companies such as Alpha Video, Echo Bridge, and Mill Creek.


In [14]:
#cell14

twowiki_test_question = "What is Calloway County High School?"
twowiki_results = retrieve_top_k_chunks(
    dataset_name="2wikimultihopqa",
    question=twowiki_test_question,
    top_k=DEFAULT_TOP_K
)

display(twowiki_results[["rank", "chunk_id", "title", "cosine_distance", "cosine_similarity"]])
print(twowiki_results.loc[0, "document"][:1000])

,rank,chunk_id,title,cosine_distance,cosine_similarity
0,1,2wikimultihopqa_chunk_00000001,Calloway County High School,0.184552,0.815448
1,2,2wikimultihopqa_chunk_00001870,Bell County High School,0.435930,0.564070
2,3,2wikimultihopqa_chunk_00001564,Cherokee High School (Georgia),0.474791,0.525209
3,4,2wikimultihopqa_chunk_00000005,Creswell High School (Oregon),0.476129,0.523871
4,5,2wikimultihopqa_chunk_00001556,Hidden Valley High School (Virginia),0.509187,0.490813
5,6,2wikimultihopqa_chunk_00000052,"DeSales High School (Louisville, Kentucky)",0.517590,0.482410


Title: Calloway County High School
Text: Calloway County High School is a public high school located in Murray, Kentucky. The school was formed from the consolidation of six high schools from across the county: Hazel High School, Lynn Grove High School, Kirksey High School, Almo High School, New Concord High School, and Faxon High School.
Organizations: Clubs/Organizations
State champions: Wrestling: David Woods 195 lbs (2017) Bass Fishing: Bracken Robertson & Dillon Starks (2013) Boys Cross Country: 1984 (2A) Fast Pitch Softball: 2004 Girls Golf: 2012 (Individual, Anna Hack)
W. Earl Brown, actor: Pookie Jones, 1989 KHSAA Mr. Football winner*


In [15]:
#cell15

for dataset_name, cfg in DATASETS.items():
    print("=" * 80)
    print("Dataset:", dataset_name)
    print("DB directory:", cfg["db_dir"])
    print("Collection name:", cfg["collection_name"])

    manifest_path = cfg["db_dir"] / "build_manifest.json"
    if manifest_path.exists():
        with open(manifest_path, "r", encoding="utf-8") as f:
            manifest = json.load(f)
        print("Manifest:")
        print(json.dumps(manifest, ensure_ascii=False, indent=2))

    print("DB folder contents:")
    for item in sorted(cfg["db_dir"].iterdir()):
        print(" -", item.name)

Dataset: hotpotqa
DB directory: /content/drive/MyDrive/final_project/RAG/hotpotqa
Collection name: chunks
Manifest:
{
  "dataset": "hotpotqa",
  "source_json": "/content/drive/MyDrive/final_project/RAG/hotpotqa_docs_chunks.json",
  "db_dir": "/content/drive/MyDrive/final_project/RAG/hotpotqa",
  "collection_name": "chunks",
  "model_name": "Qwen/Qwen3-Embedding-8B",
  "embedding_dimension": 4096,
  "similarity_space": "cosine",
  "document_format": "Title: {Title}\\nText: {Text}",
  "num_input_chunks": 35029,
  "collection_count": 35029,
  "encode_batch_size": 16,
  "elapsed_seconds": 3152.9848442077637,
  "rebuilt_database": true
}
DB folder contents:
 - 3259fc64-ff95-4de3-a68b-e254173ec26e
 - build_manifest.json
 - chroma.sqlite3
Dataset: 2wikimultihopqa
DB directory: /content/drive/MyDrive/final_project/RAG/2wikimultihopqa
Collection name: chunks
Manifest:
{
  "dataset": "2wikimultihopqa",
  "source_json": "/content/drive/MyDrive/final_project/RAG/2wikimultihopqa_docs_chunks.json",
